###  批量获取文献数据

In [27]:
import pandas as pd
from Bio import Medline
from Bio import Entrez
import string 

Entrez.email = "1219131962@qq.com"
Entrez.api_key = "6fcb01c2989c35aa1da3d9d0118abedfc409"
PMID_info = []
CGM_result_dict = {}

def get_keywords(PMID_list):
    """Thank to Dr. Mao Zhitao for his help in this function.
    """
    # if 'nan' in PMID_list: PMID_list.remove('nan')
    # PMID_list = [x for x in PMID_list if str(x) != 'nan']
    
    for i in PMID_list:
        # print(i)
        handle = Entrez.efetch(db="pubmed", id=i, rettype="medline", retmode="text")
        records = Medline.parse(handle)
        records = list(records)

        for record in records:
            title = record.get("TI","?")
            abstract = record.get("AB","?")
            keywords = record.get("OT", "?")
            Journal = record.get("JT", "?")
            country = record.get("AD", "?")[0].split(', ')[-1].strip(string.punctuation)
            year = record.get("DP","?")
            pmid = record.get('PMID','?')

            PMID_info.append({'PMID': pmid,
                            'Title':title,
                            'Abstract': abstract,
                            'Keywords': keywords,
                            'Journal': Journal,
                            'Country': country,
                            'Year': year})
            
    for key, value in CGM_result_dict.items():
        PMID_info.append([key,value])

    return PMID_info

In [28]:
df = pd.read_csv('pmid-Myceliopht-set.txt', sep='\t', header=None, names=['PMID'])

# 如果某行"PMID"列的值为空值，那么这行数据将被删除。
df = df.dropna(subset=['PMID'])
df = df.reset_index(drop=True)

# 将df第1到100行另存为一个dataframe, 避免api被封
df1 = df.iloc[0:100,:]
df2 = df.iloc[100:200,:]
df3 = df.iloc[200:]

In [ ]:
df2

In [30]:
# 根据PMID获取文章信息：关键词、摘要、标题、年份、国家、期刊，不过国家这里效果不太理想
get_keywords(list(df2.iloc[:, 0]))

df_info = pd.DataFrame(PMID_info)

In [38]:
# 读取mtd/code/Get_Paper_info/Mt_Paper.xlsx
mt_paper = pd.read_excel('Mt_Paper.xlsx')

# 删除Keywords列中为？的行
mt_paper = mt_paper[mt_paper.Keywords != '?']

mt_paper.to_excel('Mt_Paper_filter.xlsx', index=False, sheet_name='Sheet1')